### set up Jenkins in K8s

#### Install jenkins on unbutu server
```bash
 # install java on ubuntu
sudo apt update
sudo apt install openjdk-17-jdk -y

 # check java has installed with the right version
 java -version

 # install jenkins
 curl -fsSL https://pkg.jenkins.io/debian-stable/jenkins.io.key | sudo tee \\
 /usr/share/keyrings/jenkins-keyring.asc > /dev/null

 echo deb \[signed-by=/usr/share/keyrings/jenkins-keyring.asc] \\
 https://pkg.jenkins.io/debian-stable binary/ | sudo tee \\
 /etc/apt/sources.list.d/jenkins.list > /dev/null

 sudo apt update
 sudo apt install jenkins -y

 sudo systemctl start jenkins
 sudo systemctl enable jenkins
 ```

#### Set up jenkins server
* access jenkins
  + access jenkins web interface by `http://<server ip>:8080`
  + get initial admin password by `sudo cat /var/lib/jenkins/secrets/initialAdminPassword`

* Generate SSH key of bitbucket/github for jenkins
  + on jenkins server run the following commands:
  + `ssh-keygen -t ed25519` When prompt the file to save key, use /home/<your_home_folder>/.ssh/id\_ed25519
  + the public and private key files are id\_ed25519.pub and id\_ed25519, respectively
* Add public key to bitbucket
  + copy public key file's content by `cat /home/<your_home_folder>/.ssh/id_ed25519.pub`
  + go to bitbucket -> settings -> add SSH key -> add workspace key and paste the public key
* Add private key from jenkins GUI
  + in jenkins GUI -> manage Jenkins -> Credentials -> add SSH Username with private key
    + input 'git' in Username
    + in Private Key -> Enter directly, copy and paste the entire content of id\_ed25519 include ---begin and --end lines
* (Optional) copy and paste .ssh folder to /var/lib/jenkins/.ssh (not recommended) refer to [link](Set_up_Jenkins_Locally_on_Ubuntu.md)
* generate host key of github or bitbucket
```bash
    # Example of github and bitbucket
    sudo -u jenkins ssh-keyscan github.com >> known_hosts
    ssh-keyscan bitbucket.org >> known_hosts
```
* add host key to Jenkins
  + Open Jenkins UI
  + Log in to your Jenkins instance
  + Go to Manage Jenkins (usually in the left sidebar)
    + click Security
    + find and click Git Host Key Verification Configuration
 
* in Configure Host Key Verification
  + Select "Manually provided keys" as the strategy
  + In the Approved Host Keys section, paste the line you copied from ssh-keyscan, click Save
    + You might see output in known_hosts like:
    ```text
    bitbucket.org:22 SSH-2.0-OpenSSH_8.2p1 Ubuntu-4ubuntu0.5
    bitbucket.org ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIE...
    ```
    Only copy this line: `bitbucket.org ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIE...`
* test using a pipeline and build it
  ```groovy
  pipeline {
      agent any

      stages {
          stage('Checkout') {
              steps {
                  git credentialsId: 'bitbucket_ssh',
                  url: 'git@bitbucket.org:<org>/<repo name>.git', //git repo url copied from git/bitbucket repo 
                  branch: 'main'
              }
          }

          stage('Verify') {
              steps {
                  sh 'ls -la'
              }
          }
      }
  }
  ```

* install python3 and pip on jenkins server and worker nodes
  ```bash
  sudo apt update
  sudo apt install -y python3 python3-pip
  sudo apt install python3-venv
  ```

* set up worker nodes
  + create a folder as remote root directory. Permission should be given to the user who operates the worker node, usually is your user name and group when logging into the worker node machine
    ```bash
    sudo mkdir /home/jenkins # create a root directory where worker node will run tasks in the folder
    sudo chown <your_login_username>:<your_usergroup> /home/jenkins

    # install java
    sudo apt install openjdk-21-jdk
    sudo cp agent.jar /home/jenkins
    ```

* add worker node to jenkins
  + from jenkins -> manage Jenkins -> nodes -> new nodes
  + fill in the Node name (test), type (permanent Agent) -> create
  + fill in description, Number of Executors, remote root directory and labels (group work nodes), usage (use as much as possible), Launch method (by connecting it to the controller), and availability (keep online as much as possible) -> Save
  + click the created node, you will see Agent <node name>. e.g. test
  + copy the command under "Run from agent command line: (Unix), for example: 
  + ```bash
     echo <secret> > secret-file
     curl -sO http://<server ip>:8080/jnlpJars/agent.jar
     java -jar agent.jar -url http://<server ip>:8080/ -secret @secret-file -name <node name> -webSocket -workDir "/home/jenkins"
     ```
  + write down the secret string and then create system service file in /etc/systemd/system/ folder named jenkins-agent.service by
     `sudo vi /etc/systemd/system/jenkins-agent.service`
  + content of this file is in this [link](jenkins-agent_service.txt)
  + Note the file name in worker node should be jenkins-agent.service
    

#### Set up pipeline from jenkins GUI using docker agent
* define pipeline script from jenkins GUI
* jenkins installed local has agent: {docker {image: 'python:3.10'}}, any and label 'worker node name' options 
* make sure the worker node has installed a virtual env with all packages
* define worker node in agent { label 'worker node name'}
 
```groovy
pipeline {
  agent {
    docker {
      image 'python:3.10'
    }
  }

    environment {
        VENV_DIR = 'venv'
    }

    stages {
      stage('Run') {
        steps {
          sh 'python --version'
        }
      }

      stage('Git Checkout') {
        steps {
          git credentialsId: 'bitbucket_ssh',
              url: 'git@github.com:yuanMLOps/LLM_NLP.git',
              branch: 'main'
        }
      }

      stage('Verify') {
        steps {
          sh 'ls-la'
        }
      }

      stage('Setup python Environment') {
        steps { // use . $VENV_DIR/bin/activate as source $VENV_DIR/bin/activate
          sh '''
          python3 -m venv $VENV_DIR
          . $VENV_DIR/bin/activate

          pip install -r requirements.txt
          '''
        }
      }

      stage('Run Tests') {
        steps {
          sh '''
          . venv/bin/activate 
          which pytest
          export PYTHONPATH=$(pwd)
          pytest
          '''
        }
      }

    }
  }
 ```

* refer the following link for jenkins project configuration
  [jenkins project config](setup_k8s_agent.docx)        

#### Set up pipeline by jenkinsfile in code repo using kubernetes agent
* define jenkinsfile in the root folder of code repo
  + don't need to include checkout stage in jenkinsfile since it will automatically check out the code
  + other parts are the same as pipeline script defined in jenkins GUI
  + you can't run jenkinsfile with docker as agent since you can't specify which docker container to run pipeline reliably
  + for locally installed jenkins server and worker nodes, we use label to refer to the specific worker node
  
```groovy
pipeline {
  agent {
    label 'airflow_worker' // or label 'test' depends on the name of the worker node you set up in jenkins
  }

    environment {
        VENV_DIR = 'venv'
    }

    stages {
      stage('Run') {
        steps {
          sh 'python --version'
        }
      }

     stage('Verify') {
       steps {
          sh 'ls-la'
        }
      }

      //had better to pre-install the virtual env so that you don't need to install it every build run
      stage('Setup python Environment') {
        steps { // use . $VENV_DIR/bin/activate as source $VENV_DIR/bin/activate
          sh '''
          python3 -m venv $VENV_DIR
          . $VENV_DIR/bin/activate

          pip install -r requirements.txt
          '''
        }
      }

      stage('Run Tests') {
        steps {
          sh '''
          . venv/bin/activate 
          which pytest
          export PYTHONPATH=$(pwd)
          pytest
          '''
        }
      }

    }
  }
 ```
* for configuration of project in jenkins, [refer to this link](setup_k8s_jenkinsfile.docx)
* for step by step instruction to set up jenkinsfile pipeline, [refer to this link](Jenkinsfile_Setup_Guide.txt)